In [3]:
pip install pandas numpy scikit-learn tensorflow yfinance

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import yfinance as yf

def fetch_real_time_data(ticker):
    stock = yf.Ticker(ticker)
    data = stock.history(period="1d", interval="1m")  
    return data
ticker = "AAPL"
real_time_data = fetch_real_time_data(ticker)
print(real_time_data.head())

                                 Open        High         Low       Close  \
Datetime                                                                    
2025-03-19 09:30:00-04:00  214.220001  214.690002  213.910095  214.689896   
2025-03-19 09:31:00-04:00  214.675003  214.679993  214.110107  214.600006   
2025-03-19 09:32:00-04:00  214.604996  215.025406  214.570007  214.970001   
2025-03-19 09:33:00-04:00  214.964996  215.350006  214.780106  215.145004   
2025-03-19 09:34:00-04:00  215.145004  215.179993  214.730103  215.009995   

                            Volume  Dividends  Stock Splits  
Datetime                                                     
2025-03-19 09:30:00-04:00  1282838        0.0           0.0  
2025-03-19 09:31:00-04:00   104498        0.0           0.0  
2025-03-19 09:32:00-04:00   207145        0.0           0.0  
2025-03-19 09:33:00-04:00   170822        0.0           0.0  
2025-03-19 09:34:00-04:00   190719        0.0           0.0  


In [ ]:
import pandas as pd
import numpy as np

def preprocess_data(data):
    data.fillna(method='ffill', inplace=True)
    data['MA_10'] = data['Close'].rolling(window=10).mean()
    data['MA_50'] = data['Close'].rolling(window=50).mean()
    data.dropna(inplace=True)   
    return data
preprocessed_data = preprocess_data(real_time_data)
print(preprocessed_data.head())

                                 Open        High         Low       Close  \
Datetime                                                                    
2025-03-19 10:19:00-04:00  217.839996  218.089996  217.800003  217.979996   
2025-03-19 10:20:00-04:00  217.960007  218.000000  217.669998  217.895004   
2025-03-19 10:21:00-04:00  217.919998  217.960007  217.580002  217.669998   
2025-03-19 10:22:00-04:00  217.679901  217.740005  217.432205  217.550003   
2025-03-19 10:23:00-04:00  217.559998  217.559998  217.289993  217.324997   

                           Volume  Dividends  Stock Splits       MA_10  \
Datetime                                                                 
2025-03-19 10:19:00-04:00  180913        0.0           0.0  218.119431   
2025-03-19 10:20:00-04:00  119564        0.0           0.0  218.058931   
2025-03-19 10:21:00-04:00  119228        0.0           0.0  218.006430   
2025-03-19 10:22:00-04:00   90020        0.0           0.0  217.951430   
2025-03-19 10:23

C:\Users\Welcome\AppData\Local\Temp\ipykernel_27556\1070828332.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='ffill', inplace=True)


In [ ]:
import numpy as np
import time
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import yfinance as yf
def create_model(input_shape):
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dense(25))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model
def train_model(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data['Close'].values.reshape(-1, 1))
    print(f"Scaled data shape: {scaled_data.shape}") 
    training_data_len = int(np.ceil(len(scaled_data) * 0.9))
    train_data = scaled_data[0:int(training_data_len), :]
    print(f"Train data shape: {train_data.shape}")   
    if len(train_data) < 60:
        raise ValueError("Insufficient training data: At least 60 samples are required.") 
    x_train = []
    y_train = []    
    for i in range(60, len(train_data)):
        x_train.append(train_data[i-60:i, 0])
        y_train.append(train_data[i, 0])   
    x_train, y_train = np.array(x_train), np.array(y_train)
    print(f"x_train shape before reshape: {x_train.shape}") 
    x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
    print(f"x_train shape after reshape: {x_train.shape}")    
    model = create_model((x_train.shape[1], 1))
    model.fit(x_train, y_train, batch_size=1, epochs=1)    
    return model, scaler
def predict_real_time(model, scaler, data):
    if len(data['Close']) < 60:
        raise ValueError("Insufficient data: At least 60 samples are required for prediction.")    
    last_60_days = data['Close'][-60:].values
    last_60_days_scaled = scaler.transform(last_60_days.reshape(-1, 1))   
    X_test = []
    X_test.append(last_60_days_scaled)
    X_test = np.array(X_test)
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
    pred_price = model.predict(X_test)
    pred_price = scaler.inverse_transform(pred_price)  
    return pred_price[0][0]
def trading_bot(ticker):
    while True:
        data = fetch_real_time_data(ticker)
        preprocessed_data = preprocess_data(data)
        if len(preprocessed_data['Close']) < 60:
            print("Waiting for more data...")
            time.sleep(60)
            continue   
        predicted_price = predict_real_time(model, scaler, preprocessed_data)        
        current_price = preprocessed_data['Close'].iloc[-1]
        if predicted_price > current_price:
            print(f"Buy Signal: Predicted Price = {predicted_price}, Current Price = {current_price}")
        else:
            print(f"Sell Signal: Predicted Price = {predicted_price}, Current Price = {current_price}")        
        time.sleep(60)
def fetch_real_time_data(ticker):
    data = yf.download(ticker, period="120d", interval="1d")
    print(f"Fetched data shape: {data.shape}") 
    print(data.head())
    return data
def preprocess_data(data):
    if 'Close' not in data.columns:
        raise ValueError("Input data must contain a 'Close' column.")
    data['Close'].ffill(inplace=True)
    print(f"Preprocessed data shape: {data.shape}") 
    print(data.head())   
    return data
if __name__ == "__main__":
    ticker = "AAPL" 
    data = fetch_real_time_data(ticker)
    print(f"Fetched data shape: {data.shape}")  
    print(data.head())    
    preprocessed_data = preprocess_data(data)
    print(f"Preprocessed data shape: {preprocessed_data.shape}")
    model, scaler = train_model(preprocessed_data)
    trading_bot(ticker)

[*********************100%***********************]  1 of 1 completed
C:\Users\Welcome\AppData\Local\Temp\ipykernel_23168\3500380830.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Close'].ffill(inplace=True)


Fetched data shape: (120, 5)
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-09-25  225.872879  226.790857  223.528049  224.436039  42308700
2024-09-26  227.020355  227.998199  224.914988  226.800837  36636700
2024-09-27  227.289749  229.015961  226.800835  227.958291  34026000
2024-09-30  232.488327  232.488327  229.145678  229.534821  54541900
2024-10-01  225.713242  229.145674  223.248665  229.015970  63285000
Fetched data shape: (120, 5)
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-09-25  225.872879  226.790857  223.528049  224.436039  42308700
2024-09-26  227.020355  227.998199  224.914988  226.800837  36636700
2024-09-27  227.289749  229.015961  226.80083

c:\Users\Welcome\anaconda3.7\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.1345


[*********************100%***********************]  1 of 1 completed
C:\Users\Welcome\AppData\Local\Temp\ipykernel_23168\3500380830.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Close'].ffill(inplace=True)


Fetched data shape: (120, 5)
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-09-25  225.872879  226.790857  223.528049  224.436039  42308700
2024-09-26  227.020355  227.998199  224.914988  226.800837  36636700
2024-09-27  227.289749  229.015961  226.800835  227.958291  34026000
2024-09-30  232.488327  232.488327  229.145678  229.534821  54541900
2024-10-01  225.713242  229.145674  223.248665  229.015970  63285000
Preprocessed data shape: (120, 5)
Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-09-25  225.872879  226.790857  223.528049  224.436039  42308700
2024-09-26  227.020355  227.998199  224.914988  226.800837  36636700
2024-09-27  227.289749  229.015961  226.

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().